# 05 - Structure Signal Model (MobileNetV3)

Train a multi-task pore detection + texture regularity scorer for the **structure** skin signal.

## Architecture
- Backbone: MobileNetV3-Large (pretrained ImageNet)
- Multi-task head: pore count regression, texture regularity score [0-100], overall structure score [0-100]

## Preprocessing Pipeline
- Green channel isolation (best pore contrast)
- CLAHE (Contrast Limited Adaptive Histogram Equalization)
- 2D FFT for frequency-domain texture features
- LoG (Laplacian of Gaussian) blob detection for pore candidates

## Training
- Weighted MSE loss [0.3, 0.35, 0.35]
- AdamW lr=1e-4, CosineAnnealing, 30 epochs, batch=32

## Output
- ONNX model for backend inference
- Model checkpoint saved to Google Drive

In [ ]:
# Install dependencies (Colab)
!pip install -q torch torchvision timm albumentations onnx onnxruntime opencv-python scikit-image

In [ ]:
# Mount Google Drive for data access and saving results
import os

IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/cornell-hackathon/ml'
    DATA_BASE = os.path.join(DRIVE_BASE, 'data', 'structure')
    SAVE_DIR = os.path.join(DRIVE_BASE, 'models', 'structure')
else:
    # Local / VM paths
    DATA_BASE = '/root/cornell-hackathon/ml/data/structure'
    SAVE_DIR = '/root/cornell-hackathon/ml/models/structure'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Data directory: {DATA_BASE}')
print(f'Save directory: {SAVE_DIR}')

In [ ]:
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.feature import blob_log
from tqdm import tqdm
import matplotlib.pyplot as plt
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Preprocessing Functions

Green channel isolation provides the best contrast for pore detection. CLAHE enhances local contrast,
and LoG blob detection identifies pore candidates.

In [ ]:
def extract_green_channel(image: np.ndarray) -> np.ndarray:
    """Isolate green channel for best pore contrast."""
    return image[:, :, 1]


def apply_clahe(gray: np.ndarray, clip_limit: float = 3.0, tile_size: int = 8) -> np.ndarray:
    """Apply CLAHE for local contrast enhancement."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    return clahe.apply(gray)


def compute_fft_features(gray: np.ndarray) -> np.ndarray:
    """Compute 2D FFT magnitude spectrum for texture frequency analysis."""
    f_transform = np.fft.fft2(gray.astype(np.float32))
    f_shift = np.fft.fftshift(f_transform)
    magnitude = np.log1p(np.abs(f_shift))
    return (magnitude / magnitude.max() * 255).astype(np.uint8)


def detect_pores_log(gray: np.ndarray, min_sigma: float = 1, max_sigma: float = 5, threshold: float = 0.05) -> int:
    """Detect pore candidates via Laplacian of Gaussian blob detection."""
    blobs = blob_log(gray, min_sigma=min_sigma, max_sigma=max_sigma, num_sigma=5, threshold=threshold)
    return len(blobs)


def preprocess_structure(image: np.ndarray) -> dict:
    """Full preprocessing pipeline for structure signal."""
    green = extract_green_channel(image)
    enhanced = apply_clahe(green)
    fft_feat = compute_fft_features(enhanced)
    pore_count = detect_pores_log(enhanced)
    return {'enhanced': enhanced, 'fft': fft_feat, 'pore_count': pore_count}

## Dataset

Loads from prepared annotation JSONs (list format) with train/val/test splits.
Each annotation: `{image, pore_count, texture_regularity, structure_score}`

In [ ]:
class StructureDataset(Dataset):
    """Dataset for structure signal training with multi-task labels."""

    def __init__(self, image_dir: str, annotations_path: str, transform=None):
        self.image_dir = image_dir
        with open(annotations_path) as f:
            self.annotations = json.load(f)
        self.transform = transform
        # Filter out entries where image doesn't exist
        valid = []
        for ann in self.annotations:
            img_path = os.path.join(self.image_dir, ann['image'])
            if os.path.exists(img_path):
                valid.append(ann)
        print(f'Loaded {len(valid)}/{len(self.annotations)} valid samples from {annotations_path}')
        self.annotations = valid

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]

        image = cv2.imread(os.path.join(self.image_dir, ann['image']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Apply preprocessing: green channel + CLAHE
        green = extract_green_channel(image)
        enhanced = apply_clahe(green)
        # Stack as 3-channel for pretrained backbone
        image_input = np.stack([enhanced, enhanced, enhanced], axis=-1)

        if self.transform:
            augmented = self.transform(image=image_input)
            image_input = augmented['image']

        labels = torch.tensor([
            ann['pore_count'],
            ann['texture_regularity'],
            ann['structure_score'],
        ], dtype=torch.float32)

        return image_input, labels


train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [ ]:
# Load datasets
IMAGE_DIR = os.path.join(DATA_BASE, 'images')
TRAIN_ANN = os.path.join(DATA_BASE, 'annotations', 'train.json')
VAL_ANN = os.path.join(DATA_BASE, 'annotations', 'val.json')
TEST_ANN = os.path.join(DATA_BASE, 'annotations', 'test.json')

print(f'Image dir: {IMAGE_DIR}')
print(f'Train annotations: {TRAIN_ANN}')

train_ds = StructureDataset(IMAGE_DIR, TRAIN_ANN, transform=train_transform)
val_ds = StructureDataset(IMAGE_DIR, VAL_ANN, transform=val_transform)
test_ds = StructureDataset(IMAGE_DIR, TEST_ANN, transform=val_transform)

print(f'\nDataset sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

# Quick sanity check
img, lbl = train_ds[0]
print(f'Image tensor shape: {img.shape}, Labels: pore={lbl[0]:.1f}, texture={lbl[1]:.1f}, structure={lbl[2]:.1f}')

## Model Definition

MobileNetV3-Large backbone with a multi-task head producing three outputs:
pore count, texture regularity, and overall structure score.

In [ ]:
class StructureModel(nn.Module):
    """Multi-task structure signal model with MobileNetV3-Large backbone."""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=pretrained, num_classes=0)
        feat_dim = self.backbone.num_features

        self.shared_fc = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Task-specific heads
        self.pore_head = nn.Linear(256, 1)       # Pore count regression
        self.texture_head = nn.Linear(256, 1)    # Texture regularity [0-100]
        self.structure_head = nn.Linear(256, 1)  # Overall structure score [0-100]

    def forward(self, x):
        features = self.backbone(x)
        shared = self.shared_fc(features)
        pore_count = self.pore_head(shared)
        texture_reg = self.texture_head(shared)
        structure_score = self.structure_head(shared)
        return torch.cat([pore_count, texture_reg, structure_score], dim=-1)


model = StructureModel(pretrained=True).to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## Training Loop

Multi-task loss with weighted MSE for each output head. Pore count uses lower weight
since its scale is different from the normalized scores.

In [ ]:
# Hyperparameters
NUM_EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-4
LOSS_WEIGHTS = [0.3, 0.35, 0.35]  # pore, texture, structure


def train_one_epoch(model, loader, optimizer, loss_weights):
    model.train()
    total_loss = 0.0
    for images, labels in tqdm(loader, desc='Training'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        preds = model(images)

        # Weighted multi-task MSE loss
        loss = sum(
            w * nn.functional.mse_loss(preds[:, i], labels[:, i])
            for i, w in enumerate(loss_weights)
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        preds = model(images)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    mae = torch.abs(preds - labels).mean(dim=0)
    return {'pore_mae': mae[0].item(), 'texture_mae': mae[1].item(), 'structure_mae': mae[2].item()}

In [ ]:
# Create data loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Optimizer and scheduler
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f'Training for {NUM_EPOCHS} epochs with batch_size={BATCH_SIZE}, lr={LR}')
print(f'Loss weights: pore={LOSS_WEIGHTS[0]}, texture={LOSS_WEIGHTS[1]}, structure={LOSS_WEIGHTS[2]}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Training loop
best_mae = float('inf')
history = {'train_loss': [], 'pore_mae': [], 'texture_mae': [], 'structure_mae': []}

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, LOSS_WEIGHTS)
    metrics = evaluate(model, val_loader)
    scheduler.step()

    # Track history
    history['train_loss'].append(train_loss)
    history['pore_mae'].append(metrics['pore_mae'])
    history['texture_mae'].append(metrics['texture_mae'])
    history['structure_mae'].append(metrics['structure_mae'])

    avg_mae = (metrics['texture_mae'] + metrics['structure_mae']) / 2
    elapsed = time.time() - t0

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | Loss: {train_loss:.4f} | "
          f"Pore MAE: {metrics['pore_mae']:.2f} | Texture MAE: {metrics['texture_mae']:.2f} | "
          f"Structure MAE: {metrics['structure_mae']:.2f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    if avg_mae < best_mae:
        best_mae = avg_mae
        best_path = os.path.join(SAVE_DIR, 'best_structure_model.pt')
        torch.save(model.state_dict(), best_path)
        print(f'  -> Saved best model (avg score MAE: {best_mae:.2f}) to {best_path}')

print(f'\nTraining complete. Best avg MAE: {best_mae:.2f}')

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curves
axes[1].plot(history['pore_mae'], label='Pore Count MAE')
axes[1].plot(history['texture_mae'], label='Texture Regularity MAE')
axes[1].plot(history['structure_mae'], label='Structure Score MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Validation MAE per Task')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(fig_path, dpi=150)
plt.show()
print(f'Saved training curves to {fig_path}')

## Test Set Evaluation

In [ ]:
# Load best model and evaluate on test set
best_path = os.path.join(SAVE_DIR, 'best_structure_model.pt')
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
print(f'Loaded best model from {best_path}')

test_metrics = evaluate(model, test_loader)
print(f"\nTest Results:")
print(f"  Pore Count MAE:        {test_metrics['pore_mae']:.2f}")
print(f"  Texture Regularity MAE: {test_metrics['texture_mae']:.2f}")
print(f"  Structure Score MAE:    {test_metrics['structure_mae']:.2f}")

# Save test metrics
metrics_path = os.path.join(SAVE_DIR, 'test_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f'\nSaved test metrics to {metrics_path}')

## ONNX Export

Export the trained model to ONNX format for backend inference.

In [ ]:
def export_to_onnx(model, output_path):
    model.eval()
    dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        input_names=['image'],
        output_names=['scores'],
        dynamic_axes={'image': {0: 'batch'}, 'scores': {0: 'batch'}},
        opset_version=17,
    )
    print(f'Exported ONNX model to {output_path}')

    # Verify
    import onnxruntime as ort
    session = ort.InferenceSession(output_path)
    dummy_np = dummy_input.cpu().numpy()
    result = session.run(None, {'image': dummy_np})
    print(f'ONNX output shape: {result[0].shape}')
    print(f'Sample output: pore={result[0][0][0]:.2f}, texture={result[0][0][1]:.2f}, structure={result[0][0][2]:.2f}')
    return session


onnx_path = os.path.join(SAVE_DIR, 'structure_model.onnx')
ort_session = export_to_onnx(model, onnx_path)
print(f'\nONNX model size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB')

## Inference Example

In [ ]:
# Quick inference test with a sample image
import onnxruntime as ort

sample_img, sample_labels = test_ds[0]
sample_np = sample_img.unsqueeze(0).numpy()

ort_session = ort.InferenceSession(onnx_path)
ort_result = ort_session.run(None, {'image': sample_np})

print('ONNX Inference on sample image:')
print(f'  Predicted - pore_count: {ort_result[0][0][0]:.1f}, texture_regularity: {ort_result[0][0][1]:.1f}, structure_score: {ort_result[0][0][2]:.1f}')
print(f'  Ground truth - pore_count: {sample_labels[0]:.1f}, texture_regularity: {sample_labels[1]:.1f}, structure_score: {sample_labels[2]:.1f}')

In [ ]:
# Summary of saved artifacts
print('=== Saved Artifacts ===')
for fname in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f'  {fname}: {size_mb:.1f} MB')
print(f'\nAll artifacts saved to: {SAVE_DIR}')